# Imvo project: all GPU work on Colab

**Use this in the browser** (colab.research.google.com): downloads do not work from the VS Code Colab extension.

1. **Runtime → Change runtime type → T4 GPU**.
2. Run all cells. Cell 2 asks for `colab/colab_bundle.zip` (make it with `python -m src.embedding.make_colab_bundle`).
3. The last cell downloads `embeddings_all.zip` (and `results_all.zip` if you ran the analysis cell). Unzip both into the project folder.

Total GPU time on a T4: ~40 min for LaBSE (4 spelling variants) + ~15 min for the second encoder.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
import os
import glob

# 1. Locate the 'colab_bundle' folder anywhere inside /kaggle/input
bundle_matches = glob.glob("/kaggle/input/**/colab_bundle", recursive=True)

# Fallback in case Kaggle renamed the folder: search for the directory containing 'src'
if not bundle_matches:
    src_matches = glob.glob("/kaggle/input/**/src", recursive=True)
    if src_matches:
        bundle_matches = [os.path.dirname(src_matches[0])]

if bundle_matches:
    bundle_dir = bundle_matches[0]
    print(f"Loading bundle contents from: {bundle_dir}")
    
    # 2. Copy the entire contents of the bundle directly to /kaggle/working/
    !cp -r "{bundle_dir}"/* /kaggle/working/
else:
    print("Error: Could not locate the uploaded bundle folder in /kaggle/input.")

# 3. Set current working directory to top-level working folder
os.chdir("/kaggle/working")

# 4. Check workspace
!ls src data/processed
!ls labels 2>/dev/null || echo "no labels bundled (provisional threshold will be used)"

Loading bundle contents from: /kaggle/input/datasets/tawana7/colab-bundle
data/processed:
chunks.parquet	normaliser_candidates.parquet

src:
build_lexicon.py  embed.py		prepare_data.py  synthetic_check.py
classify.py	  make_colab_bundle.py	qualitative.py	 tool_metrics.py
common.py	  normalise.py		run_all.py
compare.py	  pairs.py		smoke_test.py
no labels bundled (provisional threshold will be used)


In [3]:
!pip -q install sentence-transformers pyarrow tabulate
# regenerate the modernised variants from the shipped candidate table (deterministic)
!for r in all asp prefix; do python -m src.prepare.normalise apply --rules $r; done

modern: 9.1% of isiXhosa tokens changed ({'asp': 289816, 'prefix_common': 797, 'prefix_proper': 47517}); 96.3% of chunks; thresholds 2,5 [provisional (no labelled words yet)]
modern_asp: 7.8% of isiXhosa tokens changed ({'asp': 289816}); 95.0% of chunks; thresholds 2,5 [provisional (no labelled words yet)]
modern_prefix: 1.3% of isiXhosa tokens changed ({'prefix_common': 797, 'prefix_proper': 47517}); 44.5% of chunks; thresholds 2,5 [provisional (no labelled words yet)]


## LaBSE: raw + three spelling variants
Modern variants reuse raw embeddings for unchanged chunks (all English text).

In [4]:
!python -m src.embedding.embed --variant raw --batch 128 --shard 8192
!python -m src.embedding.embed --variant modern --chunks data/processed/chunks_modern.parquet --reuse raw --batch 128 --shard 8192
!python -m src.embedding.embed --variant modern_asp --chunks data/processed/chunks_modern_asp.parquet --reuse raw --batch 128 --shard 8192
!python -m src.embedding.embed --variant modern_prefix --chunks data/processed/chunks_modern_prefix.parquet --reuse raw --batch 128 --shard 8192

config_sentence_transformers.json: 100%|████████| 122/122 [00:00<00:00, 628kB/s]
README.md: 100%|███████████████████████████| 2.02k/2.02k [00:00<00:00, 4.83MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 236kB/s]
config.json: 100%|█████████████████████████████| 804/804 [00:00<00:00, 3.41MB/s]

model.safetensors: downloading bytes: ██▌                   |  224MB, 15.5MB/s  
model.safetensors: reconstructing file:   4%|▏     | 67.1MB / 1.88GB            
model.safetensors: downloading bytes: ████████▍             |  721MB, 58.2MB/s  
model.safetensors: downloading bytes: █████████▋            |  827MB, 68.8MB/s  
model.safetensors: downloading bytes: █████████████         | 1.12GB, 90.8MB/s  
model.safetensors: downloading bytes: ██████████████▋       | 1.26GB,  101MB/s  
model.safetensors: downloading bytes: ████████████████▏     | 1.38GB,  111MB/s  
model.safetensors: downloading bytes: ████████████████▉     | 1.45GB,  116MB/s  
model.safetensors: download

## Second encoder (robustness): multilingual-e5-base, raw + modern

In [5]:
E5 = 'intfloat/multilingual-e5-base'
!python -m src.embedding.embed --variant raw --model {E5} --prefix 'query: ' --batch 128 --shard 8192
!python -m src.embedding.embed --variant modern --chunks data/processed/chunks_modern.parquet --model {E5} --prefix 'query: ' --reuse raw --batch 128 --shard 8192

modules.json: 100%|████████████████████████████| 387/387 [00:00<00:00, 1.28MB/s]
README.md: 100%|██████████████████████████████| 179k/179k [00:00<00:00, 143MB/s]
sentence_bert_config.json: 100%|██████████████| 57.0/57.0 [00:00<00:00, 189kB/s]
config.json: 100%|█████████████████████████████| 694/694 [00:00<00:00, 2.73MB/s]

model.safetensors: downloading bytes: ██▌                   |  133MB, 9.05MB/s  
model.safetensors: downloading bytes: ████▏                 |  212MB, 17.3MB/s  
model.safetensors: downloading bytes: █████▎                |  270MB, 21.5MB/s  
model.safetensors: downloading bytes: ███████▏              |  363MB, 29.3MB/s  
model.safetensors: downloading bytes: ████████▉             |  453MB, 36.7MB/s  
model.safetensors: downloading bytes: ██████████▉           |  556MB, 45.3MB/s  
model.safetensors: downloading bytes: ███████████▉          |  602MB, 49.6MB/s  
model.safetensors: downloading bytes: █████████████▉        |  702MB, 57.5MB/s  
model.safetensors: download

In [6]:
import numpy as np, glob
for f in sorted(glob.glob('data/embeddings/*.npy')):
    e = np.load(f); n = np.linalg.norm(e.astype('float32'), axis=1)
    print(f, e.shape, 'norms', n.min().round(3), n.max().round(3))

data/embeddings/modern_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_asp_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_multilingual-e5-base.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_prefix_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/raw_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/raw_multilingual-e5-base.npy (68392, 768) norms 1.0 1.0


## Optional: run the analysis here too
Labels are optional: if `labels/pairs_labelled.csv` is in the bundle it is used; otherwise the provisional `THRESHOLD` below is used for LaBSE. 
E5 similarities sit in a different range, so the E5 run uses the threshold that keeps the same share of its candidates as LaBSE keeps at `THRESHOLD` (`pairs.py match`); run the LaBSE line first. 
The LaBSE run also repeats the raw classifier with stricter pairs (0.65) into `results/raw_thr0.65`. The E5 run analyses raw + modern only (the variants embedded above).

In [7]:
THRESHOLD = 0.55   # provisional; ignored if labels/pairs_labelled.csv was bundled
!python -m src.run_all analyse --threshold {THRESHOLD}
!python -m src.run_all analyse --model-tag multilingual-e5-base --threshold {THRESHOLD}


$ python src/pairs.py candidates --variant raw --model-tag LaBSE
8,066 mutual-NN candidates from 511 issues (splits embedded: ['test', 'train', 'val'])
        count   mean    std    min    25%    50%    75%    max
split                                                         
test   1243.0  0.509  0.080  0.328  0.451  0.493  0.550  0.792
train  5618.0  0.509  0.079  0.272  0.454  0.494  0.548  0.875
val    1205.0  0.511  0.077  0.297  0.458  0.492  0.549  0.821

$ python src/pairs.py candidates --variant modern --model-tag LaBSE
8,197 mutual-NN candidates from 511 issues (splits embedded: ['test', 'train', 'val'])
        count   mean    std    min    25%    50%    75%    max
split                                                         
test   1239.0  0.515  0.083  0.334  0.455  0.497  0.560  0.816
train  5717.0  0.513  0.081  0.272  0.456  0.497  0.553  0.875
val    1241.0  0.515  0.080  0.339  0.460  0.497  0.558  0.831

$ python src/pairs.py candidates --variant modern_asp --mode

In [9]:
# 1. Create the zip archives
!cd data/embeddings && zip -q -0 ../../embeddings_all.zip *.npy *_ids.txt
!zip -q -r results_all.zip results 2>/dev/null; ls -lh *.zip

# 2. Display clickable HTML download links for Kaggle
import os
from IPython.display import FileLink, display

display(FileLink('embeddings_all.zip'))
if os.path.exists('results_all.zip'):
    display(FileLink('results_all.zip'))

-rw-r--r-- 1 root root 608M Oct 21 13:18 embeddings_all.zip
-rw-r--r-- 1 root root  40M Oct 21 13:18 results_all.zip


/kaggle/working/embeddings_all.zip

/kaggle/working/results_all.zip